In [1]:
import torch
import torch.nn as nn
import numpy as np

In [2]:
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim, max_action=1.0):
        super(Actor, self).__init__()
        
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
            nn.Tanh() # Restricts the output strictly between -1.0 and 1.0
            )
        self.max_action = max_action

In [3]:
    def forward(self, state):
        # Scale output to match the environment's maximum action range
        return self.max_action * self.fc(state)

In [4]:
class Critic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(Critic, self).__init__()
        # The critic takes BOTH state and action as inputs
        self.l1 = nn.Linear(state_dim + action_dim, 64)
        self.l2 = nn.Linear(64, 1)
    def forward(self, state, action):
        # Concatenate state and action tensors together
        inputs = torch.cat([state, action], dim=-1)
        x = torch.relu(self.l1(inputs))
        return self.l2(x)

In [5]:
def soft_update(target_net, local_net, tau=0.005):
    # Blend local weights slowly into target weights
    for target_param, local_param in zip(target_net.parameters(), local_net.parameters()):
        target_param.data.copy_(tau * local_param.data + (1.0 - tau) * target_param.data)

In [6]:
class OUNoise:
    def __init__(self, size, mu=0.0, theta=0.15, sigma=0.2):
        self.size = size
        self.mu = mu
        self.theta = theta
        self.sigma = sigma
        self.state = np.ones(size) * mu
    
    def sample(self):
        # Generate correlated continuous noise
        
        dx = self.theta * (self.mu - self.state) + self.sigma * np.random.randn(self.size)
        self.state += dx
        return self.state

In [7]:
actor = Actor(state_dim=3, action_dim=1, max_action=2.0)
target_actor = Actor(state_dim=3, action_dim=1, max_action=2.0)
critic = Critic(state_dim=3, action_dim=1)

# Copy weights initially
soft_update(target_actor, actor, tau=1.0)
# Simulate soft update over target parameters
soft_update(target_actor, actor, tau=0.005)

# Initialize exploration noise
noise = OUNoise(size=1)
print("Continuous Exploration Noise sample:", noise.sample())
print("DDPG Continuous Actor-Critic setup successfully initialized!")

Continuous Exploration Noise sample: [-0.0561289]
DDPG Continuous Actor-Critic setup successfully initialized!
